# OpenPlaque — LCX continuous gap bridge
Focused source-CCTA test of whether a continuous coronary-sized lumen bridges the prior 14–17 mm serial-QC gap. Research use only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse controls: True=reuse valid cache; False=force recompute/update.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_BRIDGE_SEARCH = True
REUSE_FIGURES = True
REUSE_REPORT = True

In [ ]:
# Step 3 — dependencies
!pip -q install scipy pandas matplotlib

In [ ]:
# Step 4 — clone the focused branch
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch lcx-gap-bridge-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque

In [ ]:
# Step 5 — initialize workflow
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.lcx_gap_bridge import LCXGapBridgeWorkflow
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'frozen_geometry': REUSE_FROZEN_GEOMETRY,
    'bridge_search': REUSE_BRIDGE_SEARCH,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LCXGapBridgeWorkflow(reuse=reuse)
display(wf.cache_status())

In [ ]:
# Step 6 — load source CCTA and freeze source/target geometry
wf.load_source_ct()
geometry = wf.load_frozen_geometry()
geometry

In [ ]:
# Step 7 — continuous component-by-component bridge search
summary = wf.search_bridge(max_bridge_mm=8.5, beam_width=40)
display(summary)
display(wf.candidates.head(12) if wf.candidates is not None else None)

In [ ]:
# Step 8 — QC figures
wf.make_figures()
from IPython.display import display, Image
for name in ['01_source_target_context.png','02_bridge_cross_sections.png','03_bridge_vs_prior_course.png','04_bridge_diagnostics.png']:
    display(Image(filename=str(wf.out / name)))

In [ ]:
# Step 9 — package report back to Drive
report = wf.make_report()
zip_path = wf.package()
print('Report:', report)
print('Report-back ZIP:', zip_path)